In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import numpy as np
import tensorflow as tf
import h5py
from scipy.ndimage import zoom
import matplotlib.pyplot as plt

file_path = '/content/drive/MyDrive/ns_incom_inhom_2d_512-0sample0.h5'
with h5py.File(file_path, 'r') as f:
    velocity_full = f['velocity'][:20]
    time_array = f['t'][:20]

velocity_train = velocity_full[:10]
time_train = time_array[:10]
velocity_test = velocity_full[10:]
time_test = time_array[10:]

downsample_factor = 128 / 512
velocity_train_small = zoom(velocity_train, (1, downsample_factor, downsample_factor, 1))  # (10, 128, 128, 2)
velocity_test_small = zoom(velocity_test, (1, downsample_factor, downsample_factor, 1))    # (10, 128, 128, 2)

grid_size = 128
x = np.linspace(0, 1, grid_size)
y = np.linspace(0, 1, grid_size)
X, Y = np.meshgrid(x, y, indexing='ij')
spatial_coords = np.stack([X.ravel(), Y.ravel()], axis=1)

num_timesteps = velocity_train_small.shape[0]
num_points = spatial_coords.shape[0]
inputs_train = np.hstack([
    np.tile(spatial_coords, (num_timesteps, 1)),
    np.repeat(time_train, num_points)[:, None]
]).astype(np.float32)

def fourier_features(x, L=1.0, num_frequencies=6):
    frequencies = 2.0 ** np.arange(num_frequencies) * np.pi / L
    x_proj = x[..., None] * frequencies
    x_encoded = np.concatenate([np.sin(x_proj), np.cos(x_proj)], axis=-1)
    return x_encoded.reshape(x.shape[0], -1)

inputs_encoded = fourier_features(inputs_train, L=1.0, num_frequencies=6).astype(np.float32)
outputs = velocity_train_small.reshape(-1, 2).astype(np.float32)

num_samples = 10_000
indices = np.random.choice(inputs_encoded.shape[0], size=num_samples, replace=False)
inputs_small = inputs_encoded[indices]
outputs_small = outputs[indices]

batch_size = 128
dataset = tf.data.Dataset.from_tensor_slices((inputs_small, outputs_small)).shuffle(10000).batch(batch_size)

# Model
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(inputs_small.shape[1],)),
    tf.keras.layers.Dense(128, activation='tanh'),
    tf.keras.layers.Dense(128, activation='tanh'),
    tf.keras.layers.Dense(128, activation='tanh'),
    tf.keras.layers.Dense(128, activation='tanh'),
    tf.keras.layers.Dense(128, activation='tanh'),
    tf.keras.layers.Dense(3)  # u, v, p
])

# Loss Function
def pinn_loss(model, x, viscosity=tf.constant(0.01, dtype=tf.float32)):
    with tf.GradientTape(persistent=True) as tape2:
        tape2.watch(x)
        with tf.GradientTape(persistent=True) as tape1:
            tape1.watch(x)
            pred = model(x)
            u, v, p = pred[:, 0:1], pred[:, 1:2], pred[:, 2:3]

        u_x, u_y, u_t = tape1.gradient(u, x)[:, 0:1], tape1.gradient(u, x)[:, 1:2], tape1.gradient(u, x)[:, 2:3]
        v_x, v_y, v_t = tape1.gradient(v, x)[:, 0:1], tape1.gradient(v, x)[:, 1:2], tape1.gradient(v, x)[:, 2:3]
        p_x, p_y = tape1.gradient(p, x)[:, 0:1], tape1.gradient(p, x)[:, 1:2]
    u_xx = tape2.gradient(u_x, x)[:, 0:1]
    u_yy = tape2.gradient(u_y, x)[:, 1:2]
    v_xx = tape2.gradient(v_x, x)[:, 0:1]
    v_yy = tape2.gradient(v_y, x)[:, 1:2]

    continuity = u_x + v_y
    momentum_u = u_t + u * u_x + v * u_y + p_x - viscosity * (u_xx + u_yy)
    momentum_v = v_t + u * v_x + v * v_y + p_y - viscosity * (v_xx + v_yy)

    return tf.reduce_mean(tf.square(continuity)) + \
           tf.reduce_mean(tf.square(momentum_u)) + \
           tf.reduce_mean(tf.square(momentum_v))

# Training
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

@tf.function
def train_step(x, y_true):
    lambda_data = 1.0
    lambda_phys = 10.0
    with tf.GradientTape() as tape:
        y_pred = model(x, training=True)
        loss_data = tf.reduce_mean(tf.square(y_true - y_pred[:, :2]))
        loss_phys = pinn_loss(model, x)
        loss_total = lambda_data * loss_data + lambda_phys * loss_phys
    grads = tape.gradient(loss_total, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss_total, loss_data, loss_phys

for epoch in range(500):
    total_loss, data_loss_total, phys_loss_total = 0.0, 0.0, 0.0
    for step, (x_batch, y_batch) in enumerate(dataset):
        loss, data_loss, phys_loss = train_step(x_batch, y_batch)
        total_loss += loss.numpy()
        data_loss_total += data_loss.numpy()
        phys_loss_total += phys_loss.numpy()
    if epoch % 50 == 0 or epoch == 0:
        print(f"Epoch {epoch}: Total loss ={total_loss/(step+1):.8f}, Data loss ={data_loss_total/(step+1):.8f}, Phys loss ={phys_loss_total/(step+1):.8f}")

spatial_coords = np.stack([X.ravel(), Y.ravel()], axis=1)
num_test_timesteps = velocity_test_small.shape[0]
inputs_test = np.hstack([
    np.tile(spatial_coords, (num_test_timesteps, 1)),
    np.repeat(time_test, spatial_coords.shape[0])[:, None]
]).astype(np.float32)

inputs_test_encoded = fourier_features(inputs_test, L=1.0, num_frequencies=6).astype(np.float32)
outputs_test = velocity_test_small.reshape(-1, 2).astype(np.float32)

preds_test = model.predict(inputs_test_encoded, verbose=1)

# Visualization
u_true = outputs_test[:, 0].reshape(num_test_timesteps, 128, 128)[0]
v_true = outputs_test[:, 1].reshape(num_test_timesteps, 128, 128)[0]
speed_true = np.sqrt(u_true**2 + v_true**2)

u_pred = preds_test[:, 0].reshape(num_test_timesteps, 128, 128)[0]
v_pred = preds_test[:, 1].reshape(num_test_timesteps, 128, 128)[0]
speed_pred = np.sqrt(u_pred**2 + v_pred**2)

plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
plt.imshow(speed_true, origin='lower', cmap='viridis')
plt.title("Gerçek |u|")
plt.colorbar()

plt.subplot(1, 3, 2)
plt.imshow(speed_pred, origin='lower', cmap='viridis')
plt.title("Tahmin |u|")
plt.colorbar()

plt.subplot(1, 3, 3)
plt.imshow(np.abs(speed_true - speed_pred), origin='lower', cmap='hot')
plt.title("Mutlak Hata")
plt.colorbar()

plt.tight_layout()
plt.show()

Epoch 0: Total loss =0.25763124, Data loss =0.19137932, Phys loss =0.00662518


KeyboardInterrupt: 

In [4]:
all_errors = []

for t in range(num_timesteps):
    u_true = outputs[:, 0].reshape(num_timesteps, 128, 128)[t]
    v_true = outputs[:, 1].reshape(num_timesteps, 128, 128)[t]
    speed_true = np.sqrt(u_true**2 + v_true**2)

    u_pred = preds[:, 0].reshape(num_timesteps, 128, 128)[t]
    v_pred = preds[:, 1].reshape(num_timesteps, 128, 128)[t]
    speed_pred = np.sqrt(u_pred**2 + v_pred**2)

    error = np.abs(speed_true - speed_pred).flatten()  # her zaman adımındaki hataları tek diziye al
    all_errors.append(error)

plt.figure(figsize=(12, 5))
plt.boxplot(all_errors, positions=np.arange(num_timesteps), showfliers=False)
plt.title("Zaman Adımına Göre Mutlak Hata Dağılımı |u|")
plt.xlabel("Zaman Adımı")
plt.ylabel("Mutlak Hata")
plt.grid(True)
plt.show()

NameError: name 'preds' is not defined

In [5]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

fig, axs = plt.subplots(1, 2, figsize=(10, 5))
cmap = 'viridis'

im_true = axs[0].imshow(np.zeros((128, 128)), origin='lower', cmap=cmap, vmin=0, vmax=1)
axs[0].set_title("Gerçek |u|")

im_pred = axs[1].imshow(np.zeros((128, 128)), origin='lower', cmap=cmap, vmin=0, vmax=1)
axs[1].set_title("Tahmin |u|")

for ax in axs:
    ax.axis('off')

def animate(t):
    u_true = outputs[:, 0].reshape(num_timesteps, 128, 128)[t]
    v_true = outputs[:, 1].reshape(num_timesteps, 128, 128)[t]
    speed_true = np.sqrt(u_true**2 + v_true**2)

    u_pred = preds[:, 0].reshape(num_timesteps, 128, 128)[t]
    v_pred = preds[:, 1].reshape(num_timesteps, 128, 128)[t]
    speed_pred = np.sqrt(u_pred**2 + v_pred**2)

    im_true.set_data(speed_true)
    im_pred.set_data(speed_pred)
    return im_true, im_pred

ani = animation.FuncAnimation(fig, animate, frames=num_timesteps, interval=500)
plt.close(fig)

HTML(ani.to_jshtml())

NameError: name 'preds' is not defined